In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np

In [12]:
# =========================== Eliminarlo e Importarlo desde el .py =========================== #
# Definition of attributes and targets

# =========================== 21 atributos seleccionados =========================== #

selected_21_attributes = [
    'np_without_modification', 'surface_modification', 'size_tem', 'zeta_potential', 
    'incubation_protein_source', 'incubation_plasma_concentration', 'incubation_np_concentration',
    'centrifugation_speed', 'centrifugation_time', 'centrifugation_temperature', 'np_type',
    'np_shape', 'dispersion_medium', 'dispersion_medium_ph', 'size_dls', 'pdi', 'incubation_culture',
    'incubation_time', 'incubation_temperature', 'centrifugation_repetitions', 'modification_type'
]


categorical_21_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
    'np_type', 'np_shape', 'dispersion_medium', 'incubation_culture', 'modification_type'
]

# =========================== 10 atributos seleccionados =========================== #

selected_10_attributes = [
    'np_without_modification', 'surface_modification', 
    'incubation_protein_source', 'size_tem', 'zeta_potential', 'incubation_plasma_concentration', 
    'incubation_np_concentration','centrifugation_speed', 
    'centrifugation_time', 'centrifugation_temperature'    
]

categorical_10_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
]

# =========================== 17 atributos seleccionados (Jair) =========================== #

selected_17_attributes = [
    'np_without_modification', 'surface_modification', 'zeta_potential', 
    'incubation_protein_source', 'incubation_plasma_concentration', 'incubation_np_concentration',
    'np_type', 'np_shape', 'dispersion_medium', 'dispersion_medium_ph', 'size_dls', 'pdi', 
    'incubation_culture', 'incubation_time', 'incubation_temperature', 'modification_type'
]

categorical_17_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
    'np_type', 'np_shape', 'dispersion_medium', 'incubation_culture', 'modification_type'
]

# =========================== 16 atributos seleccionados (sin np_type para el entrenamiento) =========================== #

selected_16_attributes = [
    'np_without_modification', 'surface_modification', 'zeta_potential', 
    'incubation_protein_source', 'incubation_plasma_concentration', 'incubation_np_concentration', 
    'np_shape', 'dispersion_medium', 'dispersion_medium_ph', 'size_dls', 'pdi', 
    'incubation_culture', 'incubation_time', 'incubation_temperature', 'modification_type'
]

categorical_16_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
    'np_shape', 'dispersion_medium', 'incubation_culture', 'modification_type'
]

# =========================== 178 Proteinas =========================== #

protein_targets = [
    "P01871", "P01024", "P02647", "P02649", "P02768", "P04004", "P01834", 
    "P10909", "P02652", "P00734", "P01009", "P01042", "P01857", "P01859", 
    "P01023", "P01619", "P04196", "P02656", "P04114", "P02787", "P02766", 
    "P06396", "P06727", "P0C0L5", "P02671", "P08603", "P02765", "Q14624", 
    "P0DOY2", "P01008", "P68871", "P01764", "P04003", "P0C0L4", "P01876", 
    "P02749", "P02675", "P01860", "P01011", "P00747", "P02774", "P00751", 
    "P19823", "P08697", "P19827", "P69905", "P02760", "P02748", "P02679", 
    "P02790", "B9A064", "P07996", "P05155", "P01766", "P12259", "P00738",
    "P02751", "P02654", "P04406", "P00739", "P02655", "P00736", "P07225", 
    "P05154", "P09871", "P60709", "P03952", "P02747", "P05546", "P02746",
    "P27169", "P01019", "P35542", "P04217", "Q14520", "P02743", "P02763", 
    "P49908", "O43866", "P18428", "P55056", "P04264", "P00748", "P01591", 
    "P01861", "Q92954", "P02776", "P0DJI8", "P01031", "P05156", "P13671", 
    "Q03591", "P06312", "P00740", "P01615", "P00742", "P04070", "O14791", 
    "P05090", "P02745", "P00488", "Q13103", "P01877", "P01700", "P10643", 
    "Q9UK55", "P03951", "Q06033", "Q96IY4", "P13645", "P04433", "P07358",
    "P27918", "P05452", "P20851", "P07357", "P07360", "P01599", "O95445", 
    "Q9BXR6", "P02775", "P02741", "P35579", "P15169", "Q13790", "P35858", 
    "P49747", "P36955", "P19652", "P07737", "P22891", "P35908", "P80748", 
    "P08514", "P01593", "Q04756", "P06733", "P23528", "P63104", "P18065", 
    "P08519", "Q86UX7", "P02753", "Q5TB80", "P22352", "P00746", "P35443", 
    "P62937", "P10720", "P48740", "P25311", "P43652", "P35527", "Q9Y490", 
    "P05106", "P17936", "P18206", "P02788", "P06702", "P01701", "Q6Q788", 
    "Q96PD5", "Q9UGM5", "O00391", "P11142", "P14618", "P23142", "P68366",
    "P12814", "P61224", "Q13201", "P67936", "P0DOY3", "P08709", "P01034", 
    "P81605", "P02533", "P11226"]

In [10]:
# =========================== Eliminarlo e Importarlo desde el .py =========================== #

# Métricas de evaluación utilizadas en el paper original

def paper_r2(y_true, y_pred):
    """Paper's metric: Pearson correlation"""
    # Verificamos si los valores reales o las predicciones son constantes (desviación estándar = 0)
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return 0.0  # Puedes retornar 0.0 o np.nan, según lo que prefieras para tu promedio
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    return corr

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

In [11]:
def cross_validation_results(csv_filename='individual_proteins_dataset', selected_attributes=None, categorical_attributes=None, protein_targets=None, output_filename):
    
    """
    Resumen de funcionalidad:
    1. Valida los parámetros de entrada y carga el dataset desde un archivo CSV.
    2. Preprocesa los datos: selecciona los atributos de interés y aplica One-Hot Encoding a las variables categóricas.
    3. Realiza una validación cruzada de 10 pliegues (10-fold Cross-Validation) para cada proteína objetivo especificada.
    4. Entrena un modelo RandomForestRegressor (con 500 estimadores y submuestreo del 30%) en cada pliegue y evalúa su rendimiento.
    5. Calcula y reporta métricas de desempeño (R2 ajustado del paper y RMSE) por pliegue, promedios por proteína y un promedio global final.
    6. Guarda todos los resultados detallados (métricas por pliegue y promedios) en un archivo CSV de salida.
    """
    
    if selected_attributes == None or categorical_attributes == None or protein_targets == None:
        print("No se han especificado los atributos seleccionados, los Atributos Categóricos o los Atributos Objetivos")
        return;

    # Cargamos el dataset o el archivo .csv
    df = pd.read_csv(f"{csv_filename}.csv")
    
    # Seleccionamos los atributos de interes del dataset
    X = df[selected_attributes]
    
    # Hacemos la codificación One-hot a los atributos categóricos
    X = pd.get_dummies(X, columns=categorical_attributes)

    # Inicializamos el constructor de KFold 
    kf = KFold(n_splits=10, shuffle=True, random_state=42)

    # Definimos las variables donde se van a almacenar los promedios de las metricas
    all_target_r2_means = []
    all_target_rmse_means = []
    
    results = [] 
    
    for target in protein_targets:
        y = df[target].values
    
        fold_r2_scores = []
        fold_rmse_scores = []
    
        print(f"\n==================== Target {target} ====================")
    
        for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
            
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            
            # Modifique: random_state de 42 a 1234 (como se muestra en el repositorio del paper)
            # Ademas modifique: max_samples=0.3 (lo que supongo que es n_samples)
            
            model = RandomForestRegressor(n_estimators=500, random_state=42, max_samples=0.3) 
            model.fit(X_train, y_train)
    
            preds = model.predict(X_test)
    
            # Calcula las métricas R2 y RMSE para este fold
            r2_score = paper_r2(y_test, preds)
            rmse_score = rmse(y_test, preds)
           
            fold_r2_scores.append(r2_score)
            fold_rmse_scores.append(rmse_score)
    
            print(f"Fold {fold_idx}: R2={r2_score:.4f}, RMSE={rmse_score:.4f}")
    
            # Guardamos el resultado de las métricas del fold 
            results.append({
                "target": target,
                "fold": fold_idx,
                "r2": r2_score,
                "rmse": rmse_score
            })
    
        # Calculamos el promedio de las métricas para esta proteina
        mean_r2 = np.mean(fold_r2_scores)
        mean_rmse = np.mean(fold_rmse_scores)
    
        all_target_r2_means.append(mean_r2)
        all_target_rmse_means.append(mean_rmse)
    
        print(f"Mean R2 for {target}:  {mean_r2:.4f}")
        print(f"Mean RMSE for {target}: {mean_rmse:.4f}")
    
        # Guardamos el resultado promedio en una fila extra
        results.append({
            "target": target,
            "fold": "mean",
            "r2": mean_r2,
            "rmse": mean_rmse
        })

    # Calculamos el promedio final de las métricas de todas las proteinas
    final_r2_average = np.mean(all_target_r2_means)
    final_rmse_average = np.mean(all_target_rmse_means)
    
    print("\n==================== Final Results ====================")
    print(f"Final average R2:               {final_r2_average:.4f}")
    print(f"Final average RMSE:             {final_rmse_average:.4f}")
    
    df_results = pd.DataFrame(results)
    df_results.to_csv(f"{output_filename}.csv", index=False)
    
    print("\n==================== Saved ====================")
    print(f"Results written to {filename}")

In [1]:
# =========================== Cross Validation con 21 atributos (dataset completo) =========================== #

csv_filename = 'individual_proteins_dataset'
output_filename = f"{csv_filename}-cv_21a"

# Definimos la ruta de donde se encuentra el dataset
input_filepath = os.path.join('..', 'datasets', csv_filename)

# Definimos la ruta en donde se guardaran las salidas
output_dir = 'cv_results'
os.makedirs(output_dir, exist_ok=True)
output_filepath = os.path.join(output_dir, output_filename)

cross_validation_results(
    csv_filename=input_filepath,
    selected_attributes=selected_21_attributes,
    categorical_attributes=categorical_21_attributes,
    protein_targets=protein_targets,
    output_filename=output_filepath
)

NameError: name 'cross_validation_results' is not defined

In [ ]:
# =========================== Cross Validation con 10 atributos (dataset completo) =========================== #

csv_filename='individual_proteins_dataset'
output_filename=f"{csv_filename}-cv_10a"

# Definimos la ruta de donde se encuentra el dataset
input_filepath = os.path.join('..', 'datasets', csv_filename)

# Definimos la ruta en donde se guardaran las salidas
output_dir = 'cv_results'
os.makedirs(output_dir, exist_ok=True)
output_filepath = os.path.join(output_dir, output_filename)

cross_validation_results(
    csv_filename=input_filepath,
    selected_attributes=selected_10_attributes,
    categorical_attributes=categorical_10_attributes,
    protein_targets=protein_targets,
    output_filename=output_filepath
)

In [ ]:
# =========================== Cross Validation con 17 atributos (dataset completo) =========================== #

csv_filename='individual_proteins_dataset'
output_filename=f"{csv_filename}-cv_17a"

# Definimos la ruta de donde se encuentra el dataset
input_filepath = os.path.join('..', 'datasets', csv_filename)

# Definimos la ruta en donde se guardaran las salidas
output_dir = 'cv_results'
os.makedirs(output_dir, exist_ok=True)
output_filepath = os.path.join(output_dir, output_filename)

cross_validation_results(
    csv_filename=input_filepath,
    selected_attributes=selected_17_attributes,
    categorical_attributes=categorical_17_attributes,
    protein_targets=protein_targets,
    output_filename=output_filepath    
)

In [13]:
# =========================== Cross Validation con 16 atributos (dataset liposomas sin np_type) =========================== #

csv_filename = "liposomes_dataset-np_type"
output_filename=f"{csv_filename}-cv_16a"

# Definimos la ruta de donde se encuentra el dataset
input_filepath = os.path.join('..', 'datasets', csv_filename)

# Definimos la ruta en donde se guardaran las salidas
output_dir = 'cv_results'
os.makedirs(output_dir, exist_ok=True)
output_filepath = os.path.join(output_dir, output_filename)

cross_validation_results(
    csv_filename=input_filepath,
    selected_attributes=selected_16_attributes,
    categorical_attributes=categorical_16_attributes,
    protein_targets=protein_targets
    output_filename=output_filepath
)


==================== Target P01871 ====================
Fold 1: R2=0.7931, RMSE=0.5180
Fold 2: R2=0.0546, RMSE=1.8937
Fold 3: R2=0.7060, RMSE=0.8073
Fold 4: R2=0.4524, RMSE=1.3139
Fold 5: R2=0.5282, RMSE=1.0546
Fold 6: R2=0.4689, RMSE=0.6938
Fold 7: R2=0.6327, RMSE=0.9487
Fold 8: R2=0.5987, RMSE=1.1901
Fold 9: R2=0.6399, RMSE=1.1771
Fold 10: R2=0.4835, RMSE=0.7535
Mean R2 for P01871:  0.5358
Mean RMSE for P01871: 1.0351

==================== Target P01024 ====================
Fold 1: R2=0.8831, RMSE=0.8854
Fold 2: R2=0.3322, RMSE=1.1736
Fold 3: R2=0.7249, RMSE=1.4853
Fold 4: R2=0.6185, RMSE=1.6421
Fold 5: R2=0.8259, RMSE=0.9495
Fold 6: R2=0.6184, RMSE=1.5451
Fold 7: R2=0.7078, RMSE=1.4982
Fold 8: R2=0.6400, RMSE=1.2594
Fold 9: R2=0.7009, RMSE=1.5632
Fold 10: R2=0.7184, RMSE=1.5353
Mean R2 for P01024:  0.6770
Mean RMSE for P01024: 1.3537

==================== Target P02647 ====================
Fold 1: R2=0.3088, RMSE=1.3030
Fold 2: R2=0.6305, RMSE=0.8280
Fold 3: R2=0.6374, RMSE=1.5206
